In [2]:
import pandas as pd
import yaml
import sys
import yaml
import warnings
import optuna
import shap
import lightgbm as lgb
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, average_precision_score, accuracy_score, roc_auc_score, f1_score
from sklearn.calibration import CalibrationDisplay
from sklearn.model_selection import StratifiedKFold
from optuna.storages.journal import JournalStorage, JournalFileBackend

In [3]:
 path_yaml = Path("../../src/config.yaml")
try:

    with open(path_yaml, "r") as file:
        config = yaml.safe_load(file)

except FileNotFoundError:
    print("Config file not found")
    sys.exit(1)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [4]:
path_preprocessed = Path("../../data/processed/telco_preprocessed.csv")
df = pd.read_csv(path_preprocessed)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('future.no_silent_downcasting', True)

In [5]:
id_col = config["mapping"]["customer_id"]
if id_col in df.columns:
    df.set_index(id_col, inplace=True)

In [6]:
X = df.drop(columns = ["is_fiber"])
y = df["is_fiber"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
num_features = config["features"]["numerical_features"]
ord_features = [config["features"]["ordinal_categoricals"][0]]
nom_features = config["features"]["nominal_categoricals"]
ord_feat_categ = [config["features"]["ordinal_orders"][0]]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("ord", OrdinalEncoder(categories=ord_feat_categ), ord_features),
         ("nom", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), nom_features), 
    ],
    remainder="drop"
)

preprocessor.set_output(transform="pandas")

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [8]:
def test_model(model, X_t, y_t, name="Model", pos_label=1):
    prob_indices = list(model.classes_)
    pos_idx = prob_indices.index(pos_label)
    pred_prob = model.predict_proba(X_t)[:, pos_idx]
    
    pred_acc = model.predict(X_t)
    
    metrics = {
        "ROC-AUC": roc_auc_score(y_t, pred_prob),
        "PR-AUC": average_precision_score(y_t, pred_prob, pos_label=pos_label),
        "Log loss": log_loss(y_t, model.predict_proba(X_t)),
        "Brier Score": brier_score_loss(y_t, pred_prob, pos_label=pos_label),
        "Accuracy": accuracy_score(y_t, pred_acc),
        "F1-Score": f1_score(y_t, pred_acc, pos_label=pos_label)
        }

    df_report = pd.DataFrame.from_dict(metrics, orient='index', columns=[name])
    return df_report

In [9]:
def get_shap(model, X: pd.DataFrame, plot: bool =False) -> list:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer(X, check_additivity=False)
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

    feature_ranking = pd.DataFrame({
        "feature": X.columns,
        "importance": mean_abs_shap
    }).sort_values(by="importance", ascending=False)

    col_ranking = feature_ranking["feature"].tolist()
    
    if plot == True:
        plt.figure(figsize=(10,6))
        shap.plots.beeswarm(shap_values)
        plt.show()

    return col_ranking

In [10]:
default_model = lgb.LGBMClassifier()
default_model.fit(X_train, y_train)

col_imp_ranking = get_shap(default_model, X_train)

[LightGBM] [Info] Number of positive: 1439, number of negative: 2700
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000292 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 364
[LightGBM] [Info] Number of data points in the train set: 4139, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.347669 -> initscore=-0.629303
[LightGBM] [Info] Start training from score -0.629303


In [11]:
col_imp_ranking

['ord__contract_type',
 'nom__phone_service_yes',
 'num__tenure_months',
 'nom__multiple_lines_yes',
 'nom__streaming_tv_yes',
 'nom__streaming_movies_yes',
 'nom__paperless_billing_yes',
 'nom__payment_method_electronic_check',
 'nom__payment_method_mailed_check',
 'nom__device_protection_yes',
 'nom__senior_citizen_yes',
 'num__cltv',
 'nom__tech_support_yes',
 'nom__online_backup_yes',
 'nom__online_security_yes',
 'nom__dependents_yes',
 'nom__partner_yes',
 'nom__gender_male',
 'nom__payment_method_credit_card']

In [11]:
default_model.get_params()

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'num_leaves': 31,
 'objective': None,
 'random_state': None,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0}

In [12]:
X_train

,num__tenure_months,num__cltv,ord__contract_type,nom__gender_male,nom__senior_citizen_yes,nom__partner_yes,nom__dependents_yes,nom__phone_service_yes,nom__multiple_lines_yes,nom__online_security_yes,nom__online_backup_yes,nom__device_protection_yes,nom__tech_support_yes,nom__streaming_tv_yes,nom__streaming_movies_yes,nom__paperless_billing_yes,nom__payment_method_credit_card,nom__payment_method_electronic_check,nom__payment_method_mailed_check
4644,0.736522,0.586437,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
1891,-0.800253,0.389806,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2923,0.196574,-0.625833,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
83,0.238108,1.153246,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1770,-0.717184,-1.341398,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4827,-1.215598,1.253271,2.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
776,0.611918,1.119905,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
5169,1.442607,0.692447,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4190,-0.924857,0.908740,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [65]:
neg_targ = len(df[df["is_fiber"] == 0])
pos_targ = len(df) - neg_targ
scale_pos_weight = neg_targ/pos_targ

n_features = 18
X_filtered = X_train[col_imp_ranking[:n_features]]

In [66]:
def objective(trial):
    
    params = {
        'objective': 'binary',
        'metric': 'average_precision',
        'boosting_type': 'gbdt',
        'n_jobs': -1,
        'verbose': -1,
        "force_row_wise": True,
        "random_state": 42,

        'scale_pos_weight': scale_pos_weight,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 7, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
    }

    dtrain = lgb.Dataset(X_filtered, label=y_train)

    cv_results = lgb.cv(
            params,
            dtrain,
            num_boost_round=1000,
            nfold=5,
            stratified=True,
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False),
            ],
        )

    best_pr_auc = cv_results["valid average_precision-mean"][-1]

    return float(best_pr_auc)

In [67]:
sampler = optuna.samplers.GPSampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_warmup_steps=15)
storage = JournalStorage(JournalFileBackend("./optuna_journal.log"))

study = optuna.create_study(
    study_name="tuning_lightgbm_fiber5",
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    storage=storage,
    load_if_exists=True
)

default_params ={
     'colsample_bytree': 1.0,
     'learning_rate': 0.1,
     'min_child_samples': 20,
     'num_leaves': 31,
    }
if len(study.trials) == 0:
    study.enqueue_trial(default_params)

In [68]:
study.optimize(objective, show_progress_bar=True, n_trials=200)

  0%|          | 0/200 [00:00<?, ?it/s]

In [69]:
best_params = study.best_params
best_params

{'learning_rate': 0.029638366425465777,
 'num_leaves': 127,
 'min_child_samples': 64,
 'colsample_bytree': 0.5972251036724263}

In [75]:
test_params = {'learning_rate': 0.05898602410432694,
'num_leaves': 20,
'min_child_samples': 65,
'colsample_bytree': 0.6682096494749166}
lgbm_clf = lgb.LGBMClassifier(
**test_params
)

lgbm_clf.fit(X_train[col_imp_ranking[:n_features]], y_train)

,boosting_type,'gbdt'
,num_leaves,20
,max_depth,-1
,learning_rate,0.05898602410432694
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,65


In [76]:
default_results = test_model(default_model, X_test, y_test, "Default Model")
optimized_results = test_model(lgbm_clf, X_test[col_imp_ranking[:n_features]], y_test, "Optimized Model")
df_final = pd.concat([default_results, optimized_results], axis=1 )

In [77]:
df_final

,Default Model,Optimized Model
ROC-AUC,0.881675,0.890597
PR-AUC,0.775838,0.793499
Log loss,0.407771,0.389105
Brier Score,0.136396,0.129517
Accuracy,0.797101,0.803865
F1-Score,0.702550,0.708752
